## Py19. Monthly Order Summary — objective

- Identify grouping keys and the business grain of summary measures.
- Interpret selected fields, samples and counts at their actual output grain.

Historical results; no code executed.
- [Original inputs: gizmobox/orders](../../../data/gizmobox/orders/) — 4 files.

## Learning notes — corrective commentary (originals unchanged)

Supplied orders JSONL includes malformed lines; inspect corrupt-record handling before interpreting row counts. Original files are linked above.

# Py19. Monthly Order Summary

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark/Py19.%20Monthly%20Order%20Summary.json)

## Monthly Order Summary
For each of the customer, produce the following summary per month
1. total orders
1. total items bought
1. total amount spent

In [ ]:
df_orders = spark.table('gizmobox.silver.py_orders')
display(df_orders)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,8,Gaming Console,499,1,Electronics,Sony,Blue
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,4,Smartwatch,299,2,Electronics,Canon,Gray
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,7,External Hard Drive,129,3,Electronics,Dell,Blue
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,3,Wireless Headphones,199,3,Electronics,Apple,White
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,2,Laptop,999,1,Electronics,Microsoft,Black
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,5,Tablet,399,1,Electronics,LG,White
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,10,Drone,799,1,Electronics,Dell,Black
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,3,Wireless Headphones,199,1,Electronics,HP,Silver
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,1,Smartphone,699,1,Electronics,Samsung,Silver


In [ ]:
from pyspark.sql import functions as F

df_order_summary = (
    df_orders
    .withColumn("order_month", F.date_format('transaction_timestamp', 'yyyy-MM'))
    .groupBy('order_month', 'customer_id')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.sum('quantity').alias('total_items_bought'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_amount')
    )
)

display(df_order_summary)

order_month,customer_id,total_orders,total_items_bought,total_amount
2024-11,9179,1,1,999
2024-12,2054,1,2,398
2024-10,5816,2,4,1296
2025-01,9084,2,4,1796
2025-01,6384,2,3,1297
2024-11,8580,1,4,586
2024-11,2639,1,6,984
2025-01,9605,2,7,1823
2025-01,9247,1,2,258
2024-11,4914,2,2,398


In [ ]:
df_order_summary.writeTo('gizmobox.gold.py_order_summary_monthly').createOrReplace()

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.gold.py_order_summary_monthly;
```

In [ ]:
# Archived output only; no computation.

order_month,customer_id,total_orders,total_items_bought,total_amount
2024-11,9179,1,1,999
2024-12,2054,1,2,398
2024-10,5816,2,4,1296
2025-01,9084,2,4,1796
2025-01,6384,2,3,1297
2024-11,8580,1,4,586
2024-11,2639,1,6,984
2025-01,9605,2,7,1823
2025-01,9247,1,2,258
2024-11,4914,2,2,398
